# Part 2 · The same code on MNIST

A small U-Net denoiser, trained with the class training loop on the linear path, makes digits with the
class sampler. Nothing is trained here: the setup cells fetch the trained weights.

**Run:** the setup cells first, then Kernel → Restart Kernel and Run All. E7 at the end is optional.

## Setup
Run these cells once. They install the packages, write the helper code next to this notebook (you can leave the long cell collapsed), and fetch the model weights. If the kernel restarts, run them again.

In [ ]:
# @title Install the packages (already present in Colab and in the course environment)
%pip install --quiet torch numpy matplotlib


In [ ]:
%%writefile lab.py
"""Helpers for the DSC 180 D32 flow-matching lab: an S-shaped curve in 2D, and MNIST.

Course conventions (Topic 1): t = 0 is noise, t = 1 is data, and
x_t = alpha_t * x1 + sigma_t * x0. In class the path is linear (alpha_t = t,
sigma_t = 1 - t) and the network is a denoiser: it predicts the clean point x1.
"""
from __future__ import annotations

import math
import numbers
import os
import sys
from pathlib import Path

# Apple GPUs: run operations that MPS lacks on the CPU instead of failing. PyTorch
# reads this variable when it is imported, so it must be set before `import torch`.
_FALLBACK_SET_BEFORE_TORCH = ("torch" not in sys.modules
                              or os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK") == "1")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
import torch.nn.functional as F  # noqa: E402

HERE = Path(__file__).resolve().parent
COLORS = {"data": "#727CA3", "denoiser": "#700000", "ink": "#1A1A1A", "muted": "#8A8A93",
          "green": "#1F6F4A", "gold": "#B8902B", "panel": "#DDE9EC"}


# ------------------------------------------------------------------ devices
def _available(name: str) -> bool:
    if name == "cpu":
        return True
    if name == "cuda":                       # NVIDIA, and AMD GPUs through PyTorch's ROCm builds
        return torch.cuda.is_available()
    if name == "mps":                        # Apple GPUs
        backend = getattr(torch.backends, "mps", None)
        return bool(backend is not None and backend.is_available())
    if name == "xpu":                        # Intel GPUs
        xpu = getattr(torch, "xpu", None)
        return bool(xpu is not None and xpu.is_available())
    return False


def _works(device: torch.device) -> bool:
    """Run one tiny operation: a GPU that this PyTorch build cannot use fails here, not halfway through."""
    try:
        return bool((torch.ones(2, device=device) * 2).sum().item() == 4)
    except Exception:
        return False


def pick_device(prefer: str | None = None) -> torch.device:
    """Pick CUDA, then Apple MPS, then Intel XPU, then the CPU.

    `prefer` (or the LAB_DEVICE environment variable) overrides the order, for example
    pick_device("cpu") or pick_device("cuda:1"). A choice that cannot run here falls back to the CPU.
    """
    prefer = prefer or os.environ.get("LAB_DEVICE") or None
    if prefer:
        try:
            device = torch.device(str(prefer).strip().lower())
        except (RuntimeError, ValueError):
            print(f"{prefer!r} is not a device name; using CPU instead.")
            return torch.device("cpu")
        if _available(device.type) and _works(device):
            return device
        print(f"Device {prefer!r} is not available here; using CPU instead.")
        return torch.device("cpu")
    for name in ("cuda", "mps", "xpu"):
        if not _available(name):
            continue
        device = torch.device(name)
        if not _works(device):
            print(f"Found a {name} device, but PyTorch could not run on it; trying the next option.")
            continue
        if name == "mps" and not _FALLBACK_SET_BEFORE_TORCH:
            print("Tip: torch was imported before lab, so Apple's CPU fallback is off. If an "
                  "operation fails, restart the kernel and import lab before torch.")
        return device
    return torch.device("cpu")


def cap_cpu_threads(max_threads: int = 8) -> None:
    """Small networks get slower, not faster, with many CPU threads (a 128-core server is a crawl)."""
    if torch.get_num_threads() > max_threads:
        torch.set_num_threads(max_threads)


cap_cpu_threads()


def cpu_generator(seed: int) -> torch.Generator:
    """Randomness is drawn on the CPU and then moved, so a seed gives the same numbers on every device."""
    return torch.Generator().manual_seed(int(seed))


def noise(shape, seed: int, device="cpu") -> torch.Tensor:
    """Standard Gaussian noise of `shape`, drawn on the CPU from `seed`, then moved to `device`."""
    return torch.randn(tuple(shape), generator=cpu_generator(seed)).to(device)


# ------------------------------------------------------------------ the S curve
R_TOP, R_BOT = 0.8, 1.2          # a printed S: the bottom loop is larger, so the mean is off the curve
_ARC = 1.5 * math.pi             # each loop is three quarters of a circle


def _s_raw(u) -> np.ndarray:
    """Points on the unscaled S at arclength fractions u in [0, 1], from the top end to the bottom end."""
    u = np.asarray(u, dtype=np.float64)
    top_len = _ARC * R_TOP
    s = u * _ARC * (R_TOP + R_BOT)
    out = np.empty(u.shape + (2,))
    top = s < top_len
    th = s[top] / R_TOP                                  # top loop: right end, over the top, left, to the joint
    out[top, 0] = R_TOP * np.cos(th)
    out[top, 1] = R_TOP + R_TOP * np.sin(th)
    th = math.pi / 2 - (s[~top] - top_len) / R_BOT       # bottom loop: from the joint, right, under, left end
    out[~top, 0] = R_BOT * np.cos(th)
    out[~top, 1] = -R_BOT + R_BOT * np.sin(th)
    return out


CURVE_U = (np.arange(20000) + 0.5) / 20000               # arclength fractions of the reference points
_RAW = _s_raw(CURVE_U)
RAW_MEAN = _RAW.mean(0)                                  # about (0.085, -0.485): inside the bottom loop
SCALE = float(np.sqrt(((_RAW - RAW_MEAN) ** 2).sum(1).mean() / 2))
OFFSET = (4.5, 0.0)                                      # the data sit to the right of the noise, apart from it
CURVE = (_RAW - RAW_MEAN) / SCALE + np.asarray(OFFSET)   # 20,000 points evenly spaced along the S
_CURVE_SQ = (CURVE ** 2).sum(1)
XLIM, YLIM = (-3.5, 7.0), (-4.0, 4.0)                    # a view that shows the noise and the data


def sample_s_curve(n: int, generator: torch.Generator | None = None) -> torch.Tensor:
    """n training points, uniform along the S, with about unit spread around OFFSET (float32, CPU)."""
    u = torch.rand(n, generator=generator, dtype=torch.float64).numpy()
    return torch.from_numpy((_s_raw(u) - RAW_MEAN) / SCALE + np.asarray(OFFSET)).float()


def _as_points(x) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    return np.asarray(x, dtype=np.float64).reshape(-1, 2)


def _nearest(x: np.ndarray):
    """Index of, and distance to, the nearest reference point on the S for each row of x."""
    idx = np.empty(len(x), dtype=np.int64)
    dist = np.empty(len(x))
    for i in range(0, len(x), 256):
        xb = x[i:i + 256]
        d2 = (xb ** 2).sum(1)[:, None] - 2 * xb @ CURVE.T + _CURVE_SQ[None]
        idx[i:i + 256] = d2.argmin(1)
        dist[i:i + 256] = np.sqrt(np.maximum(d2.min(1), 0))
    return idx, dist


def distance_to_curve(x) -> np.ndarray:
    return _nearest(_as_points(x))[1]


def curve_position(x) -> np.ndarray:
    """Where on the S each point is closest to: 0 at the top end, 1 at the bottom end."""
    return CURVE_U[_nearest(_as_points(x))[0]]


def curve_metrics(x, bins: int = 20) -> dict:
    """How well samples match the S.

    mean_distance / median_distance: distance to the S (the median ignores a few runaway samples).
    share_on_curve: fraction within 0.3 of the S. share_outside_view: fraction outside the plotting view.
    coverage_tv: how unevenly samples cover the S (0 is even, close to 1 is a single spot)."""
    pts = _as_points(x)
    idx, dist = _nearest(pts)
    which = np.minimum((CURVE_U[idx] * bins).astype(int), bins - 1)
    hist = np.bincount(which, minlength=bins) / len(idx)
    inside = ((pts[:, 0] >= XLIM[0]) & (pts[:, 0] <= XLIM[1]) & (pts[:, 1] >= YLIM[0]) & (pts[:, 1] <= YLIM[1]))
    return {"mean_distance": round(float(dist.mean()), 4),
            "median_distance": round(float(np.median(dist)), 4),
            "share_on_curve": round(float((dist < 0.3).mean()), 6),
            "share_outside_view": round(float((~inside).mean()), 6),
            "coverage_tv": round(float(0.5 * np.abs(hist - 1 / bins).sum()), 4)}


def exact_denoiser(x, alpha: float, sigma: float, points=None) -> np.ndarray:
    """E[x1 | x_t = x] when x1 is uniform over `points` (default: the S) and x_t = alpha*x1 + sigma*x0."""
    if sigma <= 0:
        raise ValueError("sigma must be positive (t < 1)")
    pts = CURVE if points is None else np.asarray(points, dtype=np.float64).reshape(-1, 2)
    sq = (pts ** 2).sum(1)
    x = _as_points(x)
    out = np.empty_like(x)
    for i in range(0, len(x), 256):
        xb = x[i:i + 256]
        d2 = (xb ** 2).sum(1)[:, None] - 2 * alpha * xb @ pts.T + alpha ** 2 * sq[None]
        logw = -d2 / (2 * sigma ** 2)
        w = np.exp(logw - logw.max(1, keepdims=True))
        out[i:i + 256] = (w @ pts) / w.sum(1, keepdims=True)
    return out


def exact_sample(points=None, n: int = 500, steps: int = 64, seed: int = 1, record: bool = False):
    """The class sampler with the exact denoiser in place of a network (numpy, CPU)."""
    x = noise((n, 2), seed).numpy().astype(np.float64)
    ts = np.linspace(0, 1, steps + 1)
    path = [x.copy()]
    for t, s in zip(ts[:-1], ts[1:]):
        x = x + (s - t) * (exact_denoiser(x, t, 1 - t, points) - x) / (1 - t)
        path.append(x.copy())
    return (x, np.stack(path)) if record else x


# ------------------------------------------------------------------ paths and networks
def linear_path(t):
    """alpha, sigma, d alpha/dt, d sigma/dt for x_t = t * x1 + (1 - t) * x0."""
    return t, 1 - t, torch.ones_like(t), -torch.ones_like(t)


def _time_column(t, x: torch.Tensor) -> torch.Tensor:
    """t as a (batch, 1) column on x's device, from a number, a 0-d tensor, or one value per example.

    The column is a real copy, not an expanded view: with an expanded view, PyTorch 2.14 on Apple GPUs gives
    every image but the first a wrong time inside the sampling loop."""
    return torch.as_tensor(t, dtype=x.dtype, device=x.device).reshape(-1, 1).expand(x.shape[0], 1).contiguous()


class MLPDenoiser(nn.Module):
    """Predicts the clean point x1 from (x_t, t) for 2D data."""

    def __init__(self, width: int = 256, depth: int = 4):
        super().__init__()
        layers, d = [], 2 + 7                    # x (2), t, sin(k pi t) and cos(k pi t) for k = 1, 2, 3
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.SiLU()]
            d = width
        layers.append(nn.Linear(d, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        t = _time_column(t, x)
        k = torch.arange(1, 4, device=x.device, dtype=x.dtype) * math.pi
        return self.net(torch.cat([x, t, torch.sin(t * k), torch.cos(t * k)], dim=1))


class _Block(nn.Module):
    def __init__(self, cin: int, cout: int, emb: int):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.emb = nn.Linear(emb, cout)
        self.n1, self.n2 = nn.GroupNorm(8, cout), nn.GroupNorm(8, cout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, e):
        h = F.silu(self.n1(self.c1(x)))
        h = h + self.emb(e)[:, :, None, None]
        h = F.silu(self.n2(self.c2(h)))
        return h + self.skip(x)


class UNetDenoiser(nn.Module):
    """A small U-Net that predicts the clean image x1 from (x_t, t): 28x28 -> 14 -> 7 and back."""

    def __init__(self, ch: int = 48, emb: int = 128):
        super().__init__()
        self.temb = nn.Sequential(nn.Linear(1, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.d1, self.d2, self.d3 = _Block(1, ch, emb), _Block(ch, 2 * ch, emb), _Block(2 * ch, 2 * ch, emb)
        self.mid = _Block(2 * ch, 2 * ch, emb)
        self.u3, self.u2, self.u1 = _Block(4 * ch, 2 * ch, emb), _Block(4 * ch, ch, emb), _Block(2 * ch, ch, emb)
        self.out = nn.Conv2d(ch, 1, 3, padding=1)

    def forward(self, x, t):
        e = self.temb(_time_column(t, x))
        h1 = self.d1(x, e)
        h2 = self.d2(F.avg_pool2d(h1, 2), e)
        h3 = self.d3(F.avg_pool2d(h2, 2), e)
        u = self.u3(torch.cat([self.mid(h3, e), h3], 1), e)
        u = self.u2(torch.cat([F.interpolate(u, size=h2.shape[-2:], mode="nearest"), h2], 1), e)
        u = self.u1(torch.cat([F.interpolate(u, size=h1.shape[-2:], mode="nearest"), h1], 1), e)
        return self.out(u)


# ------------------------------------------------------------------ training and sampling
def train_denoiser(net, sample_x1, steps: int, *, batch: int = 512, lr: float = 2e-3,
                   t_low: float = 0.0, t_high: float = 1.0, path=linear_path, seed: int = 0,
                   ema=None, ema_decay: float = 0.999, snapshot_at=()):
    """The class training loop for any data and path. sample_x1(n, generator) returns n clean examples.

    Returns the loss of every update and {update: weights} snapshots (of `ema` if given)."""
    device = next(net.parameters()).device
    g = cpu_generator(seed)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    losses, snapshots = [], {}
    keep = ema if ema is not None else net
    for step in range(steps + 1):
        if step in snapshot_at:
            snapshots[step] = {k: v.detach().cpu().clone() for k, v in keep.state_dict().items()}
        if step == steps:
            break
        x1 = sample_x1(batch, g)
        x0 = torch.randn(x1.shape, generator=g)
        t = t_low + (t_high - t_low) * torch.rand((batch,) + (1,) * (x1.dim() - 1), generator=g)
        x1, x0, t = x1.to(device), x0.to(device), t.to(device)
        a, s = path(t)[:2]
        xt = a * x1 + s * x0
        loss = ((net(xt, t) - x1) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if ema is not None:
            d = ema_decay if step >= 1000 else 0.9
            with torch.no_grad():
                for pe, pn in zip(ema.parameters(), net.parameters()):
                    pe.mul_(d).add_(pn, alpha=1 - d)
        losses.append(float(loss.detach()))
    return losses, snapshots


def check_steps(steps) -> int:
    """Return `steps` as an int, or raise a clear error unless it is a positive integer."""
    if isinstance(steps, bool) or not isinstance(steps, numbers.Integral) or steps < 1:
        raise ValueError(f"steps must be a positive integer, got {steps!r}")
    return int(steps)


@torch.no_grad()
def euler_sample(net, x: torch.Tensor, steps, *, path=linear_path, t_start: float = 0.0,
                 record: bool = False):
    """Follow the velocity implied by the denoiser from t_start to t = 1 in `steps` Euler steps.

    With record=True also returns the states (steps + 1) and the denoiser's guesses (steps), on CPU."""
    steps = check_steps(steps)
    ts = torch.linspace(float(t_start), 1.0, steps + 1).tolist()
    states, guesses = [x.detach().cpu()], []
    for t, s in zip(ts[:-1], ts[1:]):
        tt = torch.full((x.shape[0],) + (1,) * (x.dim() - 1), t, dtype=x.dtype, device=x.device)
        x1_hat = net(x, tt)
        a, sg, da, ds = path(tt)
        x = x + (s - t) * (da * x1_hat + ds * (x - a * x1_hat) / sg)
        if record:
            states.append(x.detach().cpu())
            guesses.append(x1_hat.detach().cpu())
    return (x, states, guesses) if record else x


# ------------------------------------------------------------------ weights and MNIST files
def save_weights(net: nn.Module, file, /, **meta) -> None:
    """Save float32 CPU weights plus simple metadata (numbers and strings) to `file`."""
    torch.save({"state": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "meta": meta}, file)


def load_weights(net: nn.Module, path, device):
    """Load weights saved by save_weights onto any device. Returns (net in eval mode, meta)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    net.load_state_dict(ckpt["state"])
    return net.to(device).eval(), dict(ckpt.get("meta", {}))


def load_mnist_denoiser(device, path=None) -> UNetDenoiser:
    path = HERE / "mnist_denoiser.pt" if path is None else Path(path)
    meta = torch.load(path, map_location="cpu", weights_only=True).get("meta", {})
    return load_weights(UNetDenoiser(ch=int(meta.get("ch", 48))), path, device)[0]


def load_digits(path=None):
    """MNIST test digits in [-1, 1] with shape (n, 1, 28, 28), and their labels."""
    path = HERE / "mnist_digits.npz" if path is None else Path(path)
    d = np.load(path)
    x = torch.from_numpy(d["images"].astype(np.float32) / 255 * 2 - 1)[:, None]
    return x, torch.from_numpy(d["labels"].astype(np.int64))


# ------------------------------------------------------------------ plotting
def setup_axes(ax, title: str | None = None, xlim=XLIM, ylim=YLIM) -> None:
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_color(COLORS["muted"])
        spine.set_linewidth(0.6)
    if title:
        ax.set_title(title, fontsize=11, color=COLORS["ink"])


def draw_curve(ax, lw: float = 1.0, alpha: float = 0.35) -> None:
    ax.plot(CURVE[:, 0], CURVE[:, 1], color=COLORS["data"], lw=lw, alpha=alpha, zorder=0)


def to_unit(img) -> np.ndarray:
    """An image tensor in [-1, 1] as a numpy array in [0, 1] for imshow."""
    return img.detach().cpu().float().clamp(-1, 1).add(1).div(2).squeeze().numpy()


def image_grid(imgs: torch.Tensor, ncol: int, pad: int = 2) -> np.ndarray:
    """Tile (n, 1, 28, 28) images in [-1, 1] into one array in [0, 1], ncol images per row."""
    ims = imgs.detach().cpu().float().clamp(-1, 1).add(1).div(2)[:, 0].numpy()
    n, h, w = ims.shape
    nrow = math.ceil(n / ncol)
    out = np.ones((nrow * (h + pad) + pad, ncol * (w + pad) + pad))
    for k, im in enumerate(ims):
        r, c = divmod(k, ncol)
        out[pad + r * (h + pad): pad + r * (h + pad) + h, pad + c * (w + pad): pad + c * (w + pad) + w] = im
    return out


def plot_denoiser_field(ax, net, t: float, n: int = 150, seed: int = 3, device="cpu") -> float:
    """Noisy inputs at time t (gray), lines to the network's x1 guesses (red), exact answers (circles).
    Returns the root-mean-square distance between the network's guesses and the exact answers."""
    g = cpu_generator(seed)
    x1 = sample_s_curve(n, g)
    xt = t * x1 + (1 - t) * torch.randn(x1.shape, generator=g)
    with torch.no_grad():
        guess = net(xt.to(device), t).cpu().numpy()
    exact = exact_denoiser(xt, t, 1 - t)
    xt = xt.numpy()
    draw_curve(ax)
    ax.plot(np.stack([xt[:, 0], guess[:, 0]]), np.stack([xt[:, 1], guess[:, 1]]),
            color=COLORS["denoiser"], lw=0.5, alpha=0.4)
    ax.scatter(xt[:, 0], xt[:, 1], s=5, color=COLORS["muted"], zorder=2)
    ax.scatter(exact[:, 0], exact[:, 1], s=30, facecolors="none", edgecolors=COLORS["ink"],
               linewidths=0.6, zorder=3)
    ax.scatter(guess[:, 0], guess[:, 1], s=8, color=COLORS["denoiser"], zorder=4)
    return float(np.sqrt(((guess - exact) ** 2).sum(1).mean()))


# ------------------------------------------------------------------ any 2D data: the interface for your own data
VELOCITY_COLOR = "#3A3A44"                                # velocity arrows; gold marks guesses and the data mean
PART_COLORS = ("#5E3C99", "#E08214")                      # two labelled parts of a dataset (conditioning)


def _check_points(x, what: str) -> np.ndarray:
    pts = x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)
    pts = np.asarray(pts, dtype=np.float64)
    if pts.ndim != 2 or pts.shape[1] != 2:
        raise ValueError(f"{what} must return an array of shape (n, 2), got shape {pts.shape}")
    if not np.isfinite(pts).all():
        raise ValueError(f"{what} returned values that are not finite (nan or inf)")
    return pts


class Data2D:
    """A 2D dataset for the lab, built from your sampler.

        def my_data(n, g):                 # n points as an (n, 2) tensor or array; use g for all randomness
            ...
        data = lab.Data2D(my_data)

    The points are shifted and scaled like the S (mean at OFFSET = (4.5, 0), spread about 1 per coordinate), so they
    sit beside the noise cloud. Every 2D tool in this file accepts a Data2D:
        data.sample(n, g)             training points, placed the same way (float32 tensor)
        data.points                   20,000 reference points (numpy): the exact averages and the checks use them
        data.exact_denoiser(x, t)     the average of the clean points that could have produced x at time t
        data.metrics(samples)         how close samples are to the data, and how much of the data they cover
        data.xlim, data.ylim          a view that shows the noise and the data
    """

    def __init__(self, sampler, name: str = "my data", n_points: int = 20000, seed: int = 0,
                 place: bool = True, points=None):
        self.name, self._sampler, self._place = name, sampler, place
        raw = _check_points(sampler(n_points, cpu_generator(seed)) if points is None else points, "the sampler")
        self._mean = raw.mean(0)
        self._scale = float(np.sqrt(((raw - self._mean) ** 2).sum(1).mean() / 2))
        if self._scale < 1e-8:
            raise ValueError("the data have no spread: every point is the same")
        self.points = self._placed(raw) if place else raw
        self._small = self.points[np.random.default_rng(0).permutation(len(self.points))[:5000]]
        lo, hi = self.points.min(0), self.points.max(0)
        self.xlim = (min(XLIM[0], lo[0] - 0.6), max(XLIM[1], hi[0] + 0.6))
        self.ylim = (min(YLIM[0], lo[1] - 0.6), max(YLIM[1], hi[1] + 0.6))

    def _placed(self, raw: np.ndarray) -> np.ndarray:
        return (raw - self._mean) / self._scale + np.asarray(OFFSET) if self._place else raw

    def sample(self, n: int, generator: torch.Generator | None = None) -> torch.Tensor:
        return torch.from_numpy(self._placed(_check_points(self._sampler(n, generator), "the sampler"))).float()

    def exact_denoiser(self, x, t: float, small: bool = False) -> np.ndarray:
        return exact_denoiser(x, t, 1 - t, self._small if small else self.points)

    def nearest_distance(self, x) -> np.ndarray:
        x, ref = _as_points(x), self.points
        sq, out = (ref ** 2).sum(1), np.empty(len(x))
        for i in range(0, len(x), 256):
            xb = x[i:i + 256]
            out[i:i + 256] = np.sqrt(np.maximum(((xb ** 2).sum(1)[:, None] - 2 * xb @ ref.T + sq[None]).min(1), 0))
        return out

    def metrics(self, samples, near: float = 0.3) -> dict:
        """median_distance: distance from a sample to the nearest data point (robust to a few runaways).
        share_near_data: samples within `near` of the data. share_of_data_covered: data points (2,000 checked) with
        a sample within `near`; low means the samples pile up on part of the data. share_outside_view: runaways."""
        x = _as_points(samples)
        d = self.nearest_distance(x)
        check = self.points[:: max(1, len(self.points) // 2000)]
        x_sq = (x ** 2).sum(1)
        covered = np.sqrt(np.maximum(((check ** 2).sum(1)[:, None] - 2 * check @ x.T + x_sq[None]).min(1), 0)) < near
        inside = ((x[:, 0] >= self.xlim[0]) & (x[:, 0] <= self.xlim[1]) & (x[:, 1] >= self.ylim[0]) & (x[:, 1] <= self.ylim[1]))
        return {"median_distance": round(float(np.median(d)), 4), "mean_distance": round(float(d.mean()), 4),
                "share_near_data": round(float((d < near).mean()), 4),
                "share_of_data_covered": round(float(covered.mean()), 4),
                "share_outside_view": round(float((~inside).mean()), 4)}

    def draw(self, ax, alpha: float = 0.35) -> None:
        pts = self._small[:3000]
        ax.scatter(pts[:, 0], pts[:, 1], s=1.5, color=COLORS["data"], alpha=alpha, lw=0, zorder=0)

    def axes(self, ax, title: str | None = None) -> None:
        setup_axes(ax, title=title, xlim=self.xlim, ylim=self.ylim)
        self.draw(ax)


S = Data2D(sample_s_curve, name="the S", place=False, points=CURVE)   # the S from class, as one Data2D


def show_data(data: Data2D, ax=None):
    """Your data beside the noise it starts from, with its mean (gold star): check them before training."""
    import matplotlib.pyplot as plt
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4.6))
    data.axes(ax, title=f"{data.name}: 3,000 of the 20,000 reference points")
    z = noise((500, 2), 1).numpy()
    ax.scatter(z[:, 0], z[:, 1], s=4, color=COLORS["muted"], zorder=1)
    _mean_star(ax, data)
    return ax


def _mean_star(ax, data: Data2D, s: float = 260) -> None:
    m = data.points.mean(0)
    ax.scatter([m[0]], [m[1]], marker="*", s=s, facecolors="none", edgecolors=COLORS["gold"], linewidths=2, zorder=8)


def _time_label(ax, t: float) -> None:
    ax.text(0.03, 0.96, f"$t = {t:.2f}$", transform=ax.transAxes, va="top", fontsize=11, zorder=10,
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.85))


def _arrows(ax, data: Data2D, velocity, t: float, n: int = 15) -> None:
    gx, gy = np.meshgrid(np.linspace(data.xlim[0] + 0.4, data.xlim[1] - 0.4, n),
                         np.linspace(data.ylim[0] + 0.4, data.ylim[1] - 0.4, max(3, int(n * 0.76))))
    p = np.stack([gx.ravel(), gy.ravel()], 1)
    v = velocity(p, t)
    speed = np.linalg.norm(v, axis=1, keepdims=True) + 1e-9
    u = v / speed * 0.5 * np.tanh(speed / 2)           # direction, with a length that saturates
    ax.quiver(p[:, 0], p[:, 1], u[:, 0], u[:, 1], angles="xy", scale_units="xy", scale=1, color=VELOCITY_COLOR,
              width=0.005, headwidth=4, alpha=0.85, zorder=1)


def _input_density(points: np.ndarray, ref: np.ndarray, t: float) -> np.ndarray:
    """Density of training inputs x_t = t x1 + (1 - t) x0 at `points`, with x1 uniform over `ref`."""
    s2, sq, out = (1 - t) ** 2, (ref ** 2).sum(1), np.empty(len(points))
    for i in range(0, len(points), 4000):
        p = points[i:i + 4000]
        d2 = (p ** 2).sum(1)[:, None] - 2 * t * p @ ref.T + t * t * sq[None]
        out[i:i + 4000] = np.exp(-np.maximum(d2, 0) / (2 * s2)).mean(1) / (2 * np.pi * s2)
    return out


def _region(data: Data2D, t: float, res=(160, 122)):
    """Shading for the region holding 95% of the training inputs at time t (computed once)."""
    ref = data._small[:1000]
    g = cpu_generator(77)
    x1 = data.sample(2000, g).numpy()
    xt = t * x1 + (1 - t) * torch.randn((2000, 2), generator=g).numpy()
    level = np.quantile(_input_density(xt, ref, t), 0.05)
    gx, gy = np.meshgrid(np.linspace(*data.xlim, res[0]), np.linspace(*data.ylim, res[1]))
    dens = _input_density(np.stack([gx.ravel(), gy.ravel()], 1), ref, t).reshape(gx.shape)

    def paint(ax):
        ax.contourf(gx, gy, dens, levels=[level, dens.max() + 1], colors=["#DCEFE2"], zorder=0)
        ax.contour(gx, gy, dens, levels=[level], colors=[COLORS["green"]], linewidths=0.9, linestyles="--", zorder=0)
    return paint


def play(anim):
    """Show an animation in the notebook with play, pause and step buttons."""
    import matplotlib
    import matplotlib.pyplot as plt
    from IPython.display import HTML
    matplotlib.rcParams["animation.embed_limit"] = 100
    html = HTML(anim.to_jshtml(default_mode="loop"))
    plt.close(anim._fig)
    return html


def animate_forward(data: Data2D, n: int = 400, frames: int = 21, tracked: int = 40, seed: int = 4):
    """The forward process: each data point is paired with a noise point and moves along their segment."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation
    g = cpu_generator(seed)
    x1 = data.sample(n, g).numpy()
    x0 = torch.randn((n, 2), generator=g).numpy()
    ts = np.linspace(0, 1, frames)
    fig, ax = plt.subplots(figsize=(6, 4.6), dpi=64)

    def draw(k):
        ax.clear()
        data.axes(ax, title="forward process: noise–data pairs")
        few = slice(0, tracked)
        ax.plot(np.stack([x0[few, 0], x1[few, 0]]), np.stack([x0[few, 1], x1[few, 1]]),
                color=COLORS["muted"], lw=0.5, alpha=0.5, zorder=1)
        xt = ts[k] * x1 + (1 - ts[k]) * x0
        ax.scatter(xt[:, 0], xt[:, 1], s=4, color=COLORS["muted"], zorder=2)
        ax.scatter(xt[few, 0], xt[few, 1], s=12, color=COLORS["ink"], zorder=3)
        _time_label(ax, ts[k])
    return FuncAnimation(fig, draw, frames=range(frames), interval=130)


def animate_flow(data: Data2D, n: int = 400, frames: int = 26, seed: int = 2):
    """The flow the network must learn: the exact velocity field for this data, carrying noise to the data."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation
    _, path = exact_sample(data._small, n=n, steps=frames - 1, seed=seed, record=True)
    ts = np.linspace(0, 1, frames)
    fig, ax = plt.subplots(figsize=(6, 4.6), dpi=64)

    def velocity(p, t):
        return (data.exact_denoiser(p, t, small=True) - p) / (1 - t)

    def draw(k):
        ax.clear()
        data.axes(ax, title="the exact flow for this data")
        _arrows(ax, data, velocity, min(ts[k], 0.98))
        ax.scatter(path[k][:, 0], path[k][:, 1], s=5, color=COLORS["muted"], zorder=3)
        _mean_star(ax, data)
        _time_label(ax, ts[k])
    return FuncAnimation(fig, draw, frames=range(frames), interval=170)


def animate_sampling(net, data: Data2D, n: int = 400, steps: int = 64, seed: int = 7, device="cpu", every: int = 3):
    """The learned flow: red samples, one tracked path (black) and that sample's current guess (gold).
    One frame every `every` steps, plus the last one."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation
    _, states, guesses = euler_sample(net, noise((n, 2), seed, device), steps, record=True)
    xs, gs = torch.stack(states).numpy(), torch.stack(guesses).numpy()
    far = np.linalg.norm(xs[0] - data.points.mean(0), axis=1)
    h = int(np.argmax(far * (np.linalg.norm(xs[0], axis=1) < 2.5)))      # a long path that starts inside the cloud
    fig, ax = plt.subplots(figsize=(6, 4.6), dpi=64)

    def draw(k):
        ax.clear()
        data.axes(ax, title="sampling with the trained network")
        lo = max(0, k - 6)
        ax.plot(xs[lo:k + 1, :, 0], xs[lo:k + 1, :, 1], color=COLORS["muted"], lw=0.4, alpha=0.5, zorder=1)
        ax.scatter(xs[k, :, 0], xs[k, :, 1], s=5, color=COLORS["denoiser"], alpha=0.6, zorder=2)
        if k:
            ax.plot(gs[:k, h, 0], gs[:k, h, 1], color=COLORS["gold"], lw=0.8, zorder=3)
            ax.scatter(*gs[k - 1, h], s=55, color=COLORS["gold"], edgecolors=COLORS["ink"], zorder=5)
        ax.plot(xs[:k + 1, h, 0], xs[:k + 1, h, 1], color=COLORS["ink"], lw=2, zorder=4)
        ax.scatter(*xs[k, h], s=45, color=COLORS["ink"], zorder=5)
        _mean_star(ax, data)
        _time_label(ax, k / steps)
    shown = list(range(0, steps + 1, every)) + ([steps] if steps % every else [])
    return FuncAnimation(fig, draw, frames=shown, interval=150)


def _denoiser_panel(ax, net, data: Data2D, t: float, n: int = 70, device="cpu") -> None:
    g = cpu_generator(3)
    x1 = data.sample(n, g)
    xt = t * x1 + (1 - t) * torch.randn(x1.shape, generator=g)
    with torch.no_grad():
        guess = net(xt.to(device), t).cpu().numpy()
    exact = data.exact_denoiser(xt, t, small=True)
    xt = xt.numpy()
    ax.plot(np.stack([xt[:, 0], guess[:, 0]]), np.stack([xt[:, 1], guess[:, 1]]), color=COLORS["denoiser"],
            lw=0.5, alpha=0.4, zorder=1)
    ax.scatter(xt[:, 0], xt[:, 1], s=5, color=COLORS["muted"], zorder=2)
    ax.scatter(exact[:, 0], exact[:, 1], s=26, facecolors="none", edgecolors=COLORS["ink"], linewidths=0.6, zorder=3)
    ax.scatter(guess[:, 0], guess[:, 1], s=8, color=COLORS["denoiser"], zorder=4)


def animate_learning(snapshots: dict, data: Data2D, make_net, times=(0.2, 0.5, 0.9), n: int = 400, device="cpu"):
    """How the network learns: at each saved update, the velocity field and the denoiser at three times (green: where
    95% of training inputs land; circles: exact averages; red: the network), and the samples it gives."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation
    paint = {t: _region(data, t) for t in times}
    fig = plt.figure(figsize=(12, 4.8), dpi=56)
    gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1, 1.5], wspace=0.06, hspace=0.18, left=0.01, right=0.99,
                          top=0.86, bottom=0.01)
    vel = [fig.add_subplot(gs[0, i]) for i in range(3)]
    den = [fig.add_subplot(gs[1, i]) for i in range(3)]
    smp = fig.add_subplot(gs[:, 3])
    keys = sorted(snapshots)

    def draw(k):
        net = make_net()
        net.load_state_dict(snapshots[keys[k]])
        net = net.to(device).eval()
        for ax in vel + den + [smp]:
            ax.clear()
        for t, a, b in zip(times, vel, den):
            data.axes(a, title=f"velocity, $t = {t}$")
            paint[t](a)

            def velocity(p, tt, net=net):
                with torch.no_grad():
                    guess = net(torch.as_tensor(p, dtype=torch.float32, device=device), tt).cpu().numpy()
                return (guess - p) / (1 - tt)
            _arrows(a, data, velocity, t, n=13)
            data.axes(b, title=f"denoiser, $t = {t}$")
            paint[t](b)
            _denoiser_panel(b, net, data, t, device=device)
        data.axes(smp, title="samples")
        x = euler_sample(net, noise((n, 2), 10, device), 64).cpu().numpy()
        smp.scatter(x[:, 0], x[:, 1], s=5, color=COLORS["denoiser"], zorder=2)
        fig.suptitle(f"after {keys[k]:,} updates", fontsize=15)
    return FuncAnimation(fig, draw, frames=range(len(keys)), interval=600)


SNAPSHOT_STEPS = (0, 20, 40, 60, 80, 100, 130, 160, 200, 250, 300, 400, 500, 650, 800, 1000, 1300, 1600, 2000, 2500,
                  3000, 4000, 6000, 8000)          # dense early: most of the change happens in the first 1,000 updates


def show_window(data: Data2D, t: float, center, net=None, radius: float = 0.25, pairs: int = 400000,
                shown: int = 40, device="cpu", ax=None, zoom: bool | None = None) -> dict:
    """One small window of inputs at time t: the training pairs whose noisy input lands in it (gray), each with its
    own clean point (blue); the average of those clean points (black circle), and the network's guess (red).
    zoom: crop the view around the window and its clean points (default: only when t >= 0.8, where both are small)."""
    import matplotlib.pyplot as plt
    center = np.asarray(center, dtype=np.float64)
    g = cpu_generator(50)
    x1 = data.sample(pairs, g).numpy()
    xt = t * x1 + (1 - t) * torch.randn((pairs, 2), generator=g).numpy()
    inside = np.nonzero(((xt - center) ** 2).sum(1) < radius ** 2)[0]
    if len(inside) == 0:
        raise ValueError(f"no training input lands in this window at t = {t}: move it inside the green region")
    average = x1[inside].mean(0)
    guess = None
    if net is not None:
        with torch.no_grad():
            guess = net(torch.as_tensor(center[None], dtype=torch.float32, device=device), t).cpu().numpy()[0]
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4.6))
    zoom = t >= 0.8 if zoom is None else zoom
    data.axes(ax, title=f"one window at $t = {t}$" + (" (zoomed in)" if zoom else ""))
    _region(data, t)(ax)
    draw = inside[:shown]
    ax.plot(np.stack([xt[draw, 0], x1[draw, 0]]), np.stack([xt[draw, 1], x1[draw, 1]]), color=COLORS["data"],
            lw=0.6, alpha=0.45, zorder=2)
    ax.scatter(x1[draw, 0], x1[draw, 1], s=16, color=COLORS["data"], zorder=3)
    ax.scatter(xt[draw, 0], xt[draw, 1], s=7, color=COLORS["muted"], zorder=4)
    ax.add_patch(plt.Circle(center, radius, fill=False, ec=COLORS["ink"], lw=1.6, ls="--", zorder=5))
    ax.scatter(*average, s=260, facecolors="none", edgecolors=COLORS["ink"], linewidths=2.2, zorder=6)
    if guess is not None:
        ax.scatter(*guess, s=55, color=COLORS["denoiser"], zorder=7)
    if zoom:
        pts = np.concatenate([x1[draw], center[None] + radius, center[None] - radius, average[None]])
        mid, half = (pts.max(0) + pts.min(0)) / 2, max((pts.max(0) - pts.min(0)).max() / 2 + 0.5, 1.0)
        ax.set_xlim(mid[0] - 1.3 * half, mid[0] + 1.3 * half)
        ax.set_ylim(mid[1] - half, mid[1] + half)
    return {"pairs_in_window": int(len(inside)), "average": average.round(4).tolist(),
            "guess": None if guess is None else guess.round(4).tolist()}


def show_conditioning(data: Data2D, label, t: float = 0.1, center=None, n: int = 300, steps: int = 64,
                      radius: float = 0.25, w: float = 2.0, axes=None) -> dict:
    """Conditioning on two labelled parts of the data, with exact averages (no training).

    label(points) -> True for part A, False for part B. With a request, the denoiser averages only over that part:
    the same window gets a different average (left), and the flow from the same noise ends in that part (right).
    The diamond is guidance with weight w: from the overall average through part A's average and beyond."""
    import matplotlib.pyplot as plt
    is_a = np.asarray(label(data.points), dtype=bool)
    if is_a.all() or not is_a.any():
        raise ValueError("label must mark some, but not all, of the data as part A")
    parts = {"A": data.points[is_a][::2], "B": data.points[~is_a][::2]}
    center = t * data.points.mean(0) if center is None else np.asarray(center, dtype=np.float64)
    g = cpu_generator(50)
    x1 = data.sample(200000, g).numpy()
    xt = t * x1 + (1 - t) * torch.randn((200000, 2), generator=g).numpy()
    inside = np.nonzero(((xt - center) ** 2).sum(1) < radius ** 2)[0]
    a_in = np.asarray(label(x1[inside]), dtype=bool)
    average = {"none": x1[inside].mean(0), "A": x1[inside][a_in].mean(0), "B": x1[inside][~a_in].mean(0)}
    guided = average["none"] + w * (average["A"] - average["none"])
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    ax = axes[0]
    data.axes(ax, title=f"one window ($t = {t}$), three requests")
    for part, color in zip(("A", "B"), PART_COLORS):
        sel = inside[:60][(a_in[:60]) == (part == "A")]
        ax.plot(np.stack([xt[sel, 0], x1[sel, 0]]), np.stack([xt[sel, 1], x1[sel, 1]]), color=color, lw=0.6, alpha=0.4)
        ax.scatter(x1[sel, 0], x1[sel, 1], s=12, color=color, zorder=3)
        ax.scatter(*average[part], s=260, facecolors="none", edgecolors=color, linewidths=2.4, zorder=6)
    ax.add_patch(plt.Circle(center, radius, fill=False, ec=COLORS["ink"], lw=1.6, ls="--", zorder=5))
    ax.scatter(*average["none"], s=260, facecolors="none", edgecolors=COLORS["ink"], linewidths=2.4, zorder=6)
    ax.annotate("", xy=guided, xytext=average["none"], arrowprops=dict(arrowstyle="->", color=PART_COLORS[0], ls=":"))
    ax.scatter(*guided, marker="D", s=50, color=PART_COLORS[0], zorder=7)
    z = noise((n, 2), 23).numpy().astype(np.float64)
    share = {}
    for ax, part, color in zip(axes[1:], ("A", "B"), PART_COLORS):
        x, path = z.copy(), [z.copy()]
        ts = np.linspace(0, 1, steps + 1)
        for tt, s in zip(ts[:-1], ts[1:]):
            x = x + (s - tt) * (exact_denoiser(x, tt, 1 - tt, parts[part]) - x) / (1 - tt)
            path.append(x.copy())
        path = np.stack(path)
        data.axes(ax, title=f"request: part {part} (same noise)")
        ax.plot(path[:, ::3, 0], path[:, ::3, 1], color=color, lw=0.5, alpha=0.35, zorder=1)
        ax.scatter(z[:, 0], z[:, 1], s=4, color=COLORS["muted"], zorder=2)
        ax.scatter(x[:, 0], x[:, 1], s=6, color=color, zorder=3)
        landed_a = np.asarray(label(x), dtype=bool)
        share[part] = float(landed_a.mean() if part == "A" else (~landed_a).mean())
    return {"average": {k: v.round(4).tolist() for k, v in average.items()}, "guided": guided.round(4).tolist(),
            "share_on_requested_part": share}


In [ ]:
# @title Get the model weights (downloaded once; skipped if the files are already here)
import hashlib, urllib.request
from pathlib import Path
FILES = {
    "mnist_denoiser.pt": (
        "https://drive.google.com/uc?export=download&id=1XxGXmKmEl88neZMSgvnwH8N_LJfX64nr",
        "72ae49cb44ce5850ec3a9c2fd4c57969c7e28a0abf827f9bbb25ef9ae4e8380c"
    ),
    "mnist_digits.npz": (
        "https://drive.google.com/uc?export=download&id=1KnirO2MlPHgmFm0QM9nC-Hs-omxp43IU",
        "daa3d9cd0ad0273a1895af0324dcac1b08581cc27be783ceb62736cceddf8cd7"
    )
}
for name, (url, digest) in FILES.items():
    path = Path(name)
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != digest:
        print('downloading', name)
        urllib.request.urlretrieve(url, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
        raise RuntimeError(f'{name} does not match the course copy: delete it and run this cell again')
print('weights ready:', ', '.join(FILES))


In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # before torch: Apple GPUs fall back to CPU
import math
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import lab

DEVICE = lab.pick_device()          # NVIDIA, AMD, Apple or Intel GPU, else CPU
print("Using device:", DEVICE)
net = lab.load_mnist_denoiser(DEVICE)
digits, labels = lab.load_digits()

## 1. Forward process on images
The same line as in 2D: $x_t = t\,x_1 + (1-t)\,x_0$, now for 28×28 images.

In [ ]:
x1, x0 = digits[0], lab.noise((1, 28, 28), 0)
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
for ax, t in zip(axes, [0, 0.2, 0.4, 0.6, 0.8, 1]):
    ax.imshow(lab.to_unit(t * x1 + (1 - t) * x0), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"t = {t}"); ax.axis("off")
plt.show()

## 2. What the denoiser predicts
At high noise its guess is an average digit, as the 2D guess was the mean of the S.

In [ ]:
x1, x0 = digits[2:3].to(DEVICE), lab.noise((1, 1, 28, 28), 1, DEVICE)
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
with torch.no_grad():
    for j, t in enumerate([0.05, 0.2, 0.4, 0.6, 0.8, 0.95]):
        xt = t * x1 + (1 - t) * x0
        axes[0, j].imshow(lab.to_unit(xt), cmap="gray", vmin=0, vmax=1); axes[0, j].set_title(f"t = {t}")
        axes[1, j].imshow(lab.to_unit(net(xt, t)), cmap="gray", vmin=0, vmax=1)
for ax in axes.ravel(): ax.axis("off")
plt.show()

## 3. Sampling: the class sampler, for images

In [ ]:
@torch.no_grad()
def sample(net, n=16, steps=64, seed=1, t_start=0.0, x=None, record=False):
    """The class sampler. Starts from noise at t = 0, or from a given state x at time t_start."""
    steps = lab.check_steps(steps)                       # a clear error for 0, -1, 2.5 or "4"
    x = lab.noise((n, 1, 28, 28), seed, DEVICE) if x is None else x.to(DEVICE)
    ts = torch.linspace(t_start, 1, steps + 1)
    states, guesses = [x.cpu()], []
    for t, s in zip(ts[:-1], ts[1:]):
        x1_hat = net(x, t)
        v = (x1_hat - x) / (1 - t)                       # head for the predicted clean image
        x = x + (s - t) * v                              # one Euler step
        states.append(x.cpu()); guesses.append(x1_hat.cpu())
    return (x.cpu(), states, guesses) if record else x.cpu()

start = time.time()
xs = sample(net)
print(f"16 digits, 64 steps: {time.time() - start:.1f} s on {DEVICE}")
plt.figure(figsize=(4, 4)); plt.imshow(lab.image_grid(xs, ncol=4), cmap="gray"); plt.axis("off"); plt.show()

In [ ]:
fig, axes = plt.subplots(5, 8, figsize=(9, 6))
for r, k in enumerate([1, 2, 4, 16, 64]):
    for c, img in enumerate(sample(net, n=8, steps=k, seed=5)):
        axes[r, c].imshow(lab.to_unit(img), cmap="gray", vmin=0, vmax=1); axes[r, c].axis("off")
    axes[r, 0].set_title(f"{k} steps", fontsize=9, loc="left")
plt.show()

### E7. When is the digit decided? (optional)
For each seed, run the sampler to time $t$, then restart from the current guess with fresh noise,
$x = t\,\hat x_1 + (1-t)\,x_0'$, and finish sampling. Count by eye how many of the 32 restarts at each
time keep the original digit, fill in `same_digit`, and explain the trend using the 2D pictures.

In [ ]:
taus, seeds, restarts = [0.1, 0.3, 0.5, 0.7, 0.9], range(8), 4
for seed in seeds:
    base, states, guesses = sample(net, n=1, seed=seed, record=True)
    fig, axes = plt.subplots(len(taus), restarts + 1, figsize=(1.3 * (restarts + 1), 1.3 * len(taus)))
    for i, tau in enumerate(taus):
        k = int(round(tau * 64)); tk = k / 64
        fresh = lab.noise((restarts, 1, 28, 28), seed=1000 + 100 * seed + k)
        finals = sample(net, n=restarts, steps=64 - k, t_start=tk, x=tk * guesses[k] + (1 - tk) * fresh)
        for j, img in enumerate(torch.cat([base, finals])):
            axes[i, j].imshow(lab.to_unit(img), cmap="gray", vmin=0, vmax=1)
            axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        axes[i, 0].set_ylabel(f"t = {tk:.2f}", fontsize=8)
    axes[0, 0].set_title("original", fontsize=8); fig.suptitle(f"seed {seed}", fontsize=9)
    plt.show()

In [ ]:
same_digit = {0.1: None, 0.3: None, 0.5: None, 0.7: None, 0.9: None}   # your counts, out of 32
if all(v is not None for v in same_digit.values()):
    plt.plot(list(same_digit), [v / 32 for v in same_digit.values()], "o-")
    plt.xlabel("restart time t"); plt.ylabel("fraction that keep the digit"); plt.show()

**E7 prediction:**

**E7 what happened, and why:**